<a href="https://www.kaggle.com/code/avtnshm/ssl-geometry-analysis?scriptVersionId=299259670" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# =========================================================
# Setup
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import random
import pandas as pd
import scipy.stats as stats

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

print("Device:", device)

Device: cuda


In [2]:
# =========================================================
# Data (Correct SSL Version)
# =========================================================

class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


ssl_base_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor()
])

ssl_transform = TwoCropsTransform(ssl_base_transform)

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=ssl_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True,
    transform=T.ToTensor()
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print("SSL Data ready")

SSL Data ready


In [3]:
# =========================================================
# Encoder
# =========================================================

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)

In [4]:
# =========================================================
# BYOL
# =========================================================

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=2048, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)


class BYOL(nn.Module):
    def __init__(self):
        super().__init__()
        self.online_encoder = Encoder()
        self.online_projector = MLP(512)
        self.online_predictor = MLP(256, 512, 256)

        self.target_encoder = Encoder()
        self.target_projector = MLP(512)

        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_projector.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target(self, tau=0.996):
        for online, target in zip(self.online_encoder.parameters(),
                                  self.target_encoder.parameters()):
            target.data = tau * target.data + (1 - tau) * online.data

    def forward(self, x1, x2):
        o1 = self.online_predictor(
            self.online_projector(self.online_encoder(x1))
        )

        with torch.no_grad():
            t2 = self.target_projector(
                self.target_encoder(x2)
            )

        loss = 2 - 2 * F.cosine_similarity(o1, t2.detach(), dim=-1)
        return loss.mean()

In [5]:
# =========================================================
# SimCLR
# =========================================================

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.projector = MLP(512)

    def forward(self, x):
        z = self.projector(self.encoder(x))
        z = F.normalize(z, dim=1)
        return z


def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)

    sim = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([torch.arange(N, 2*N),
                           torch.arange(0, N)]).to(z.device)

    loss = F.cross_entropy(sim, positives)
    return loss

In [6]:
# =========================================================
# Train BYOL (Clean Production Version)
# =========================================================

import time

def train_byol(model, epochs=100):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    total_start_time = time.time()

    for epoch in range(epochs):
        epoch_start_time = time.time()
        model.train()
        total_loss = 0

        for (x1, x2), _ in train_loader:
            x1 = x1.to(device)
            x2 = x2.to(device)

            loss = model(x1, x2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            model.update_target()
            total_loss += loss.item()

        epoch_time = time.time() - epoch_start_time
        cumulative_time = time.time() - total_start_time

        print(f"[BYOL] Epoch {epoch+1:03d}/100 | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Epoch Time: {epoch_time/60:.2f} min | "
              f"Total Time: {cumulative_time/60:.2f} min")

    print("BYOL training complete.")
    return model.online_encoder


# ---- RUN ----
byol_model = BYOL()
byol_encoder = train_byol(byol_model, epochs=100)

[BYOL] Epoch 001/100 | Loss: 0.7656 | Epoch Time: 0.91 min | Total Time: 0.91 min
[BYOL] Epoch 002/100 | Loss: 0.5805 | Epoch Time: 0.89 min | Total Time: 1.81 min
[BYOL] Epoch 003/100 | Loss: 0.5149 | Epoch Time: 0.88 min | Total Time: 2.69 min
[BYOL] Epoch 004/100 | Loss: 0.4844 | Epoch Time: 0.89 min | Total Time: 3.57 min
[BYOL] Epoch 005/100 | Loss: 0.4666 | Epoch Time: 0.89 min | Total Time: 4.46 min
[BYOL] Epoch 006/100 | Loss: 0.4519 | Epoch Time: 0.88 min | Total Time: 5.34 min
[BYOL] Epoch 007/100 | Loss: 0.4451 | Epoch Time: 0.89 min | Total Time: 6.23 min
[BYOL] Epoch 008/100 | Loss: 0.4371 | Epoch Time: 0.89 min | Total Time: 7.12 min
[BYOL] Epoch 009/100 | Loss: 0.4288 | Epoch Time: 0.89 min | Total Time: 8.01 min
[BYOL] Epoch 010/100 | Loss: 0.4235 | Epoch Time: 0.87 min | Total Time: 8.88 min
[BYOL] Epoch 011/100 | Loss: 0.4216 | Epoch Time: 0.87 min | Total Time: 9.76 min
[BYOL] Epoch 012/100 | Loss: 0.4177 | Epoch Time: 0.88 min | Total Time: 10.64 min
[BYOL] Epoch 01

In [7]:
# =========================================================
# Train SimCLR (Clean Production Version)
# =========================================================

import time

def train_simclr(model, epochs=100):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    total_start_time = time.time()

    for epoch in range(epochs):
        epoch_start_time = time.time()
        model.train()
        total_loss = 0

        for (x1, x2), _ in train_loader:
            x1 = x1.to(device)
            x2 = x2.to(device)

            z1 = model(x1)
            z2 = model(x2)

            loss = nt_xent_loss(z1, z2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        epoch_time = time.time() - epoch_start_time
        cumulative_time = time.time() - total_start_time

        print(f"[SimCLR] Epoch {epoch+1:03d}/100 | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Epoch Time: {epoch_time/60:.2f} min | "
              f"Total Time: {cumulative_time/60:.2f} min")

    print("SimCLR training complete.")
    return model.encoder


# ---- RUN ----
simclr_model = SimCLR()
simclr_encoder = train_simclr(simclr_model, epochs=100)

[SimCLR] Epoch 001/100 | Loss: 4.6717 | Epoch Time: 0.93 min | Total Time: 0.93 min
[SimCLR] Epoch 002/100 | Loss: 4.4906 | Epoch Time: 0.93 min | Total Time: 1.86 min
[SimCLR] Epoch 003/100 | Loss: 4.4231 | Epoch Time: 0.92 min | Total Time: 2.78 min
[SimCLR] Epoch 004/100 | Loss: 4.3837 | Epoch Time: 0.92 min | Total Time: 3.70 min
[SimCLR] Epoch 005/100 | Loss: 4.3504 | Epoch Time: 0.93 min | Total Time: 4.62 min
[SimCLR] Epoch 006/100 | Loss: 4.3312 | Epoch Time: 0.93 min | Total Time: 5.55 min
[SimCLR] Epoch 007/100 | Loss: 4.3103 | Epoch Time: 0.92 min | Total Time: 6.48 min
[SimCLR] Epoch 008/100 | Loss: 4.2883 | Epoch Time: 0.93 min | Total Time: 7.41 min
[SimCLR] Epoch 009/100 | Loss: 4.2737 | Epoch Time: 0.92 min | Total Time: 8.33 min
[SimCLR] Epoch 010/100 | Loss: 4.2588 | Epoch Time: 0.93 min | Total Time: 9.26 min
[SimCLR] Epoch 011/100 | Loss: 4.2516 | Epoch Time: 0.93 min | Total Time: 10.19 min
[SimCLR] Epoch 012/100 | Loss: 4.2308 | Epoch Time: 0.92 min | Total Time: 

In [8]:
# =========================================================
# Jacobian Frobenius (Hutchinson)
# =========================================================

def jacobian_frobenius(model, x, n_samples=1):
    model.eval()
    x = x.clone().detach().to(device).requires_grad_(True)
    y = model(x)

    total = 0.0
    for _ in range(n_samples):
        v = torch.randn_like(y)
        Jv = torch.autograd.grad(
            outputs=y,
            inputs=x,
            grad_outputs=v,
            retain_graph=True
        )[0]
        total += Jv.pow(2).sum()

    return torch.sqrt(total / (n_samples * x.size(0)))

In [9]:
def compute_metrics(encoder):
    encoder = encoder.to(device)
    encoder.eval()

    jac_vals = []
    aug_vals = []
    noise_vals = []

    for i, (x, _) in enumerate(test_loader):
        if i == 5:  # limit for speed
            break

        x = x.to(device)

        # Jacobian
        jac = jacobian_frobenius(encoder, x)
        jac_vals.append(jac.item())

        # Aug sensitivity
        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x)
            aug_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

        # Noise robustness
        noise = torch.randn_like(x) * 0.1
        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x + noise)
            noise_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

    return np.mean(jac_vals), np.mean(aug_vals), np.mean(noise_vals)


byol_metrics = compute_metrics(byol_encoder)
simclr_metrics = compute_metrics(simclr_encoder)

print("BYOL:", byol_metrics)
print("SimCLR:", simclr_metrics)

BYOL: (np.float64(64.01713485717774), np.float64(0.0), np.float64(12.961149978637696))
SimCLR: (np.float64(86.7917709350586), np.float64(0.0), np.float64(19.404883193969727))


In [11]:
# =========================================================
# Linear Probe (Correct Version)
# =========================================================

# --- Create supervised datasets (NO two-crop transform) ---
supervised_train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=T.ToTensor()
)

supervised_test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=T.ToTensor()
)

supervised_train_loader = DataLoader(
    supervised_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

supervised_test_loader = DataLoader(
    supervised_test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)


class LinearProbe(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        with torch.no_grad():
            feat = self.encoder(x)
        return self.fc(feat)


def train_probe(encoder, epochs=10):
    model = LinearProbe(encoder).to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # ---- Train only classifier ----
    for epoch in range(epochs):
        model.train()
        for x, y in supervised_train_loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # ---- Evaluate ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in supervised_test_loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            preds = logits.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


# ---- RUN ----
byol_acc = train_probe(byol_encoder)
simclr_acc = train_probe(simclr_encoder)

print("BYOL Probe Acc:", byol_acc)
print("SimCLR Probe Acc:", simclr_acc)

BYOL Probe Acc: 0.6498
SimCLR Probe Acc: 0.6965


In [14]:
results = pd.DataFrame({
    "Method": ["BYOL", "SimCLR"],
    "Jacobian": [byol_metrics[0], simclr_metrics[0]],
    "Aug_Sensitivity": [byol_metrics[1], simclr_metrics[1]],
    "Noise_Sensitivity": [byol_metrics[2], simclr_metrics[2]],
    "Linear_Probe_Acc": [byol_acc, simclr_acc]
})

print(results)

corr, p = stats.pearsonr(results["Jacobian"], results["Noise_Sensitivity"])
print("Correlation (Jacobian vs Noise):", corr)

   Method   Jacobian  Aug_Sensitivity  Noise_Sensitivity  Linear_Probe_Acc
0    BYOL  64.017135              0.0          12.961150            0.6498
1  SimCLR  86.791771              0.0          19.404883            0.6965
Correlation (Jacobian vs Noise): 1.0


In [18]:
# =========================================================
# Tensor-based augmentation (no ToTensor inside)
# =========================================================

tensor_aug = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
])


def augmentation_sensitivity(encoder, loader, transform):
    encoder.eval()
    vals = []

    for i, (x, _) in enumerate(loader):
        if i == 5:
            break

        x = x.to(device)

        # Apply tensor augmentation correctly
        x_aug = torch.stack([
            transform(img.cpu()) for img in x
        ]).to(device)

        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x_aug)

        vals.append(torch.norm(f1 - f2, dim=1).mean().item())

    return np.mean(vals)


byol_aug = augmentation_sensitivity(
    byol_encoder,
    supervised_test_loader,
    tensor_aug
)

simclr_aug = augmentation_sensitivity(
    simclr_encoder,
    supervised_test_loader,
    tensor_aug
)

print("BYOL Aug Sensitivity:", byol_aug)
print("SimCLR Aug Sensitivity:", simclr_aug)

BYOL Aug Sensitivity: 14.789575958251953
SimCLR Aug Sensitivity: 17.248913955688476


In [19]:
# =========================================================
# Final Evaluation Cell (All Metrics Combined)
# =========================================================

# ---- Tensor-based augmentation (no ToTensor) ----
tensor_aug = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
])


def compute_metrics(encoder, loader):
    encoder = encoder.to(device)
    encoder.eval()

    jac_vals = []
    aug_vals = []
    noise_vals = []

    for i, (x, _) in enumerate(loader):
        if i == 5:  # limit batches for speed
            break

        x = x.to(device)

        # ---- Jacobian ----
        jac = jacobian_frobenius(encoder, x)
        jac_vals.append(jac.item())

        # ---- Augmentation Sensitivity ----
        x_aug = torch.stack([
            tensor_aug(img.cpu()) for img in x
        ]).to(device)

        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x_aug)
            aug_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

        # ---- Noise Sensitivity ----
        noise = torch.randn_like(x) * 0.1
        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x + noise)
            noise_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

    return (
        np.mean(jac_vals),
        np.mean(aug_vals),
        np.mean(noise_vals),
    )


# ---- Compute ----
byol_metrics = compute_metrics(byol_encoder, supervised_test_loader)
simclr_metrics = compute_metrics(simclr_encoder, supervised_test_loader)

# ---- Results Table ----
results = pd.DataFrame({
    "Method": ["BYOL", "SimCLR"],
    "Jacobian": [byol_metrics[0], simclr_metrics[0]],
    "Aug_Sensitivity": [byol_metrics[1], simclr_metrics[1]],
    "Noise_Sensitivity": [byol_metrics[2], simclr_metrics[2]],
    "Linear_Probe_Acc": [byol_acc, simclr_acc]
})

print(results)

# ---- Correlation (only illustrative with 2 points) ----
corr, p = stats.pearsonr(
    results["Jacobian"],
    results["Noise_Sensitivity"]
)

print("Correlation (Jacobian vs Noise):", corr)

   Method   Jacobian  Aug_Sensitivity  Noise_Sensitivity  Linear_Probe_Acc
0    BYOL  56.281437        14.493181          13.234402            0.6498
1  SimCLR  80.455182        17.022068          18.415830            0.6965
Correlation (Jacobian vs Noise): 1.0


## Preliminary Results (Single Seed)

- BYOL shows lower input Jacobian norm than SimCLR.
- BYOL shows lower noise sensitivity.
- BYOL shows lower augmentation sensitivity.
- SimCLR achieves slightly higher linear probe accuracy.

This suggests a potential geometry–discrimination trade-off:

Contrastive training may induce sharper (higher curvature) representations, 
while non-contrastive training may produce smoother mappings.

These findings are preliminary and require validation across multiple seeds.